read data from bronze layer

In [0]:
%run ../00-common/01.environment_config

In [0]:
bronze_table = f"{catalog_name}.{bronze_schema}.constructor"
silver_table = f"{catalog_name}.{silver_schema}.constructor"

In [0]:
constructor_df = spark.table(bronze_table)

keep only column require for analytics drop url column

In [0]:
from pyspark.sql import functions as F

selected_constructor_df = constructor_df.select(
    F.col("constructorId"),
    F.col("name"),
    F.col("nationality"),
    F.col("ingestion_date"),
    F.col("source_file")
)

standardise column with snaka case(countryName ==> country_name, data ---> race_data)

In [0]:
standardise_constructor_df = selected_constructor_df.withColumnsRenamed({
                                    "constructorId":"constructor_id",
                                    "name":"constructor_name",
                                    "raceName":"race_name",
                                })

#### Drop Duplicate Record

In [0]:
print(standardise_constructor_df.count())

In [0]:
constructor_distint_df = standardise_constructor_df.dropDuplicates(["constructor_id"]) # consturctor_id is a primary key

In [0]:
print(constructor_distint_df.count())

#### transform values of column nationality to title case

In [0]:
constructor_final_df = constructor_distint_df.withColumn("nationality",F.initcap(F.col("nationality")))
# display(races_final_df)

#### write data to silver layer

In [0]:
constructor_final_df.write.mode("overwrite").format("delta").saveAsTable(silver_table)

In [0]:
display(spark.table(silver_table))